In [ ]:
!git clone https://github.com/ostris/ai-toolkit /content/ai-toolkit
!cd /content/ai-toolkit && pip install -r requirements.txt
!pip install -q diffusers transformers accelerate

from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/ai-toolkit')
sys.path.insert(0, '/content/drive/MyDrive/liya_diplomCC')

DRIVE_ROOT = '/content/drive/MyDrive/liya_diplomCC'

In [ ]:
import subprocess

config = f'{DRIVE_ROOT}/configs/flux_lora_r16.yaml'
print("Training FLUX.1-dev LoRA r=16 on train_10k (4000 steps)...")
result = subprocess.run(
    ['python', '/content/ai-toolkit/run.py', config],
    capture_output=False,
)
print("DONE" if result.returncode == 0 else "FAILED")

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from pathlib import Path

TEST_PROMPTS = [
    "minimalist coffee shop logo, circular icon, dark green, flat vector design, white background",
    "tech startup logo, geometric hexagon, blue gradient, bold sans-serif, white background",
    "bakery logo, wheat sheaf icon, warm brown, handcrafted artisan style, white background",
    "fitness brand, lion silhouette, orange and black, bold geometric, white background",
    "law firm logo, balanced scales, navy blue, serif elegant typography, white background",
]

out_dir = f'{DRIVE_ROOT}/results/experiments/sd15_baseline'
Path(out_dir).mkdir(parents=True, exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

for i, prompt in enumerate(TEST_PROMPTS):
    imgs = pipe(
        prompt,
        negative_prompt="photorealistic, blurry, cluttered, complex background, 3D render",
        num_images_per_prompt=2,
        generator=torch.Generator().manual_seed(42),
        guidance_scale=7.5,
        num_inference_steps=30,
        height=512,
        width=512,
    ).images
    for j, img in enumerate(imgs):
        img.save(f"{out_dir}/prompt{i:02d}_v{j}.png")

del pipe; torch.cuda.empty_cache()
print(f"SD 1.5 baseline: {len(TEST_PROMPTS)*2} images \u2192 {out_dir}")

In [ ]:
from diffusers import FluxPipeline
import torch

flux_lora_path = f'{DRIVE_ROOT}/results/experiments/flux_r16'
out_dir = f'{DRIVE_ROOT}/results/experiments/flux_r16_samples'
Path(out_dir).mkdir(parents=True, exist_ok=True)

FLUX_PROMPTS = [f"LOGOIMG {p.replace(", white background", "")}" for p in TEST_PROMPTS]

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16,
).to("cuda")
pipe.load_lora_weights(flux_lora_path)
pipe.set_progress_bar_config(disable=True)

for i, prompt in enumerate(FLUX_PROMPTS):
    imgs = pipe(
        prompt,
        num_images_per_prompt=2,
        generator=torch.Generator().manual_seed(42),
        guidance_scale=3.5,
        num_inference_steps=20,
        height=512, width=512,
    ).images
    for j, img in enumerate(imgs):
        img.save(f"{out_dir}/prompt{i:02d}_v{j}.png")

del pipe; torch.cuda.empty_cache()
print(f"FLUX LoRA: {len(FLUX_PROMPTS)*2} images \u2192 {out_dir}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# UPDATE BEST_SDXL_RANK to the rank with lowest FID from Experiment 1 (notebook 07)
BEST_SDXL_RANK = 16

MODELS = {
    "SD 1.5 Baseline": f'{DRIVE_ROOT}/results/experiments/sd15_baseline',
    f"SDXL LoRA r={BEST_SDXL_RANK}": f'{DRIVE_ROOT}/results/experiments/sdxl_r{BEST_SDXL_RANK}_samples',
    "FLUX.1-dev LoRA": f'{DRIVE_ROOT}/results/experiments/flux_r16_samples',
}

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
for row in range(5):
    for col, (model_name, img_dir) in enumerate(MODELS.items()):
        img_path = f"{img_dir}/prompt{row:02d}_v0.png"
        if Path(img_path).exists():
            axes[row, col].imshow(Image.open(img_path))
        else:
            axes[row, col].text(0.5, 0.5, "Not yet\ngenerated",
                                ha='center', va='center', fontsize=8)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(model_name, fontsize=10, fontweight='bold')

plt.suptitle("Experiment 3: SD1.5 Baseline vs SDXL LoRA vs FLUX LoRA", fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/experiments/exp3_model_comparison.png', dpi=150)
plt.show()